In [ ]:
!pip install opencv-python-headless moviepy torchvision

In [2]:
import os
import cv2
import numpy as np
import torch
import torch.nn.functional as F
from torchvision import models, transforms
from moviepy.video.io.ffmpeg_tools import ffmpeg_extract_subclip
from moviepy.editor import VideoFileClip
import matplotlib.pyplot as plt

# Directory setup
input_dir = '/kaggle/input/unique-video-sports'  # Input video directory
output_dir = '/kaggle/working/trimmed-video-sports'  # Output directory for saving trimmed videos
os.makedirs(output_dir, exist_ok=True)


In [ ]:
import os
import subprocess

# Function to get video duration quickly with ffmpeg
def get_video_duration(video_path):
    try:
        result = subprocess.run(
            ["ffprobe", "-v", "error", "-show_entries", "format=duration", 
             "-of", "default=noprint_wrappers=1:nokey=1", video_path],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )
        return float(result.stdout.strip())
    except Exception as e:
        print(f"Error getting duration for {video_path}: {e}")
        return None

# Function to trim video using ffmpeg with precise time
def trim_video_ffmpeg(input_video, start_time, end_time, output_video):
    try:
        subprocess.run([
            "ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
            "-ss", str(start_time),  # Start time for trimming
            "-to", str(end_time),    # End time for trimming
            "-i", input_video,
            "-c:v", "copy",  # Copy video stream without re-encoding
            "-c:a", "copy",  # Copy audio stream without re-encoding
            output_video
        ], check=True)
        print(f'Trimmed video saved to {output_video}')
    except subprocess.CalledProcessError as e:
        print(f'Error trimming video {input_video}: {e}')

# Function to process video according to length criteria
def process_video_ffmpeg(video_path, output_dir):
    video_length = get_video_duration(video_path)
    
    if video_length is None:
        print(f"Skipping {video_path}: Could not retrieve duration.")
        return

    if video_length < 120:
        print(f'Skipping {video_path}: Video is less than 2 minutes.')
        return

    output_video = os.path.join(output_dir, os.path.basename(video_path))

    # For videos longer than 10 minutes, trim from 1 to 11 minutes
    if video_length > 600:
        print(f'Trimming {video_path} from 1 to 11 minutes (for videos >10 minutes).')
        trim_video_ffmpeg(video_path, start_time=60, end_time=min(660, video_length), output_video=output_video)
    else:
        # For videos 10 minutes or less, save as is with precise copying
        print(f'Processing {video_path} as is (<=10 minutes).')
        trim_video_ffmpeg(video_path, 0, video_length, output_video)

# Sequential processing of videos
for video_file in os.listdir(input_dir):
    if video_file.endswith('.mp4'):
        video_path = os.path.join(input_dir, video_file)
        process_video_ffmpeg(video_path, output_dir)

print('All videos processed and saved.')


In [ ]:
!pip install kaggle

In [5]:
# Move the API key to the correct location
!mkdir -p ~/.kaggle
!cp /kaggle/input/jsonkag/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json  # Ensure the API key file has the right permissions

In [ ]:
import os

# Define paths and metadata
dataset_name = "trim-video-sports"  # Give your dataset a unique name
folder_to_upload = "/kaggle/working/trimmed-video-sports"  # Path to your dataset
metadata_file_path = f"{folder_to_upload}/dataset-metadata.json"  # Full path to metadata file

# Create the folder to upload if it doesn't exist
!mkdir -p $folder_to_upload

# Check if the metadata file exists
if not os.path.exists(metadata_file_path):
    # Create a metadata file for the dataset
    with open(metadata_file_path, 'w') as f:
        f.write('{\n'
                '  "title": "Trimmed-sports-Videos",\n'
                '  "id": "tamimshadman/trim-video-sports",\n'
                '  "licenses": [{"name": "CC0-1.0"}]\n'
                '}')

# Check if the metadata file was created successfully
!ls {folder_to_upload}

# Create a new dataset using the Kaggle API with --dir-mode option (using zip)
!kaggle datasets create -p $folder_to_upload --dir-mode zip
